In [20]:
import geopandas as gpd
import pandas as pd

parquet_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads.parquet'

print("Loading Parquet file...")
gdf_roads = gpd.read_parquet(parquet_path)

print(f"\nOriginal shape: {gdf_roads.shape[0]} rows, {gdf_roads.shape[1]} columns")

# Calculate the minimum number of non-NaN values required to KEEP a column.
# If we drop >90% NaN, we want to KEEP columns that have at least 10% valid data.
min_valid_values = int(len(gdf_roads) * 0.10)

# Drop the columns (axis=1 means columns)
gdf_roads_clean = gdf_roads.dropna(axis=1, thresh=min_valid_values)

# Let's see what got removed
dropped_cols = set(gdf_roads.columns) - set(gdf_roads_clean.columns)

print(f"\nNew shape: {gdf_roads_clean.shape[0]} rows, {gdf_roads_clean.shape[1]} columns")
print(f"Removed {len(dropped_cols)} empty/sparse columns:")
for col in dropped_cols:
    print(f" - {col}")



Loading Parquet file...

Original shape: 656826 rows, 610 columns

New shape: 656826 rows, 14 columns
Removed 596 empty/sparse columns:
 - name:etymology:wikidata
 - source:oneway
 - operator:wikidata
 - old_name2
 - url
 - proposed
 - LINEARID
 - lanes:conditional
 - hgv:conditional
 - bicycle:lanes:backward
 - massgis:FEESYM
 - busway:left
 - was:bridge:movable
 - wheelchair
 - old_name:1748
 - lanes2
 - goods
 - coach
 - bicycle_road
 - amenity
 - leisure
 - mtb
 - traffic_calming
 - alt_name:etymology
 - minspeed
 - FIXME:hgv
 - ski:nordic
 - path
 - nat_name
 - name_1
 - cycleway:left
 - change:lanes
 - usage
 - railway
 - parking:both:restriction
 - cycleway:left:segregated
 - source:noname
 - source:maxweight
 - historic
 - orig_name
 - alt_name_1
 - massgis:DCAM_ID
 - private
 - area
 - distance
 - lanes:forward
 - access:lanes
 - cycleway:both:width
 - tiger:name_direction_prefix
 - embankment
 - emergency
 - cycleway:lanes:backward
 - FIXME:ref
 - construction
 - sidewalk:bot

In [31]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import LineString, Point
from shapely.ops import nearest_points
import warnings
warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
ROADS_PATH = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_clean.gpkg'
GRID_PATH = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/GRID/MA_osm_distribution_lines.geojson'
OUTPUT_PATH = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_with_grid.gpkg'

OVERLAP_BUFFER = 12        # Increased slightly to 12m for safer margins
OVERLAP_THRESHOLD = 0.80
MAX_EXTEND_DIST = 300

# --- 1. LOAD AND CLEAN ROADS ---
print("Loading roads...")
gdf_roads_clean = gpd.read_file(ROADS_PATH)

if 'highway' in gdf_roads_clean.columns:
    gdf_roads_clean = gdf_roads_clean[~gdf_roads_clean['highway'].isin(['motorway', 'motorway_link'])]
    gdf_roads_clean['highway'] = gdf_roads_clean['highway'].fillna('service').replace('', 'service')
    mapping_dict = {
        'trunk': 'primary', 'trunk_link': 'primary', 'primary_link': 'primary', 'unclassified': 'service'
    }
    gdf_roads_clean['highway'] = gdf_roads_clean['highway'].replace(mapping_dict)
    gdf_roads_clean['is_grid'] = 'no'
else:
    print("Warning: Could not find 'highway' column.")

# --- 2. LOAD GRID AND PREPARE ACCURATE LOCAL CRS ---
print("\nLoading OSM distribution lines...")
gdf_grid = gpd.read_file(GRID_PATH)

original_crs = gdf_roads_clean.crs
# Use EPSG:26986 (NAD83 / Massachusetts Mainland) for accurate meter distances in MA!
print(f"Projecting geometries to EPSG:26986 (Mass State Plane) for accurate buffering...")
gdf_roads_clean = gdf_roads_clean.to_crs("EPSG:26986")
gdf_grid = gdf_grid.to_crs("EPSG:26986")

gdf_grid['highway'] = 'osm_grid'
gdf_grid['is_grid'] = 'yes'

# --- 3. RESOLVE OVERLAPS (FIXED LOGIC) ---
print("\nDetecting grid lines that follow roads...")
roads_sindex = gdf_roads_clean.sindex

grid_to_drop_indices = []

for idx, grid_row in gdf_grid.iterrows():
    grid_line = grid_row.geometry

    # Quick bounding box intersection to find nearby roads
    possible_matches_index = list(roads_sindex.intersection(grid_line.bounds))
    if not possible_matches_index:
        continue

    possible_matches = gdf_roads_clean.iloc[possible_matches_index]

    # Filter to roads that actually intersect a buffered bounding box of the grid line
    grid_buffer = grid_line.buffer(OVERLAP_BUFFER)
    nearby_roads = possible_matches[possible_matches.intersects(grid_buffer)]

    if nearby_roads.empty:
        continue

    # MERGE the buffers of all nearby road segments into one continuous blob
    unioned_road_buffers = nearby_roads.geometry.buffer(OVERLAP_BUFFER).unary_union

    # How much of the grid line falls inside this merged road buffer?
    intersection = grid_line.intersection(unioned_road_buffers)
    overlap_ratio = intersection.length / grid_line.length

    if overlap_ratio >= OVERLAP_THRESHOLD:
        # The line is heavily overlapped by roads! Tag the specific roads involved.
        for road_idx in nearby_roads.index:
            # Double check that this specific road segment is close to the line
            if gdf_roads_clean.at[road_idx, 'geometry'].distance(grid_line) <= OVERLAP_BUFFER:
                gdf_roads_clean.at[road_idx, 'is_grid'] = 'yes'

        grid_to_drop_indices.append(idx)

# Drop the overlapping grid lines
gdf_grid_isolated = gdf_grid.drop(index=grid_to_drop_indices).copy()
print(f"Removed {len(grid_to_drop_indices)} grid lines that followed roads directly.")

# --- 4. EXTEND AND SNAP REMAINING GRID LINES ---
print(f"\nExtending remaining {len(gdf_grid_isolated)} grid lines along their trajectories (max {MAX_EXTEND_DIST}m)...")

def extend_line_to_road(line, roads_gdf, sindex, max_dist):
    if not isinstance(line, LineString) or len(line.coords) < 2:
        return line

    coords = list(line.coords)
    new_coords = coords.copy()

    for is_start in [True, False]:
        if is_start:
            p1, p2 = np.array(coords[1]), np.array(coords[0])
        else:
            p1, p2 = np.array(coords[-2]), np.array(coords[-1])

        vec = p2 - p1
        norm = np.linalg.norm(vec)
        if norm == 0:
            continue

        unit_vec = vec / norm
        p3 = p2 + (unit_vec * max_dist)
        search_ray = LineString([p2, p3])

        possible_matches_idx = list(sindex.intersection(search_ray.bounds))
        if possible_matches_idx:
            possible_roads = roads_gdf.iloc[possible_matches_idx]
            intersections = possible_roads.intersection(search_ray)
            valid_intersections = intersections[~intersections.is_empty]

            if not valid_intersections.empty:
                closest_pt = None
                min_dist = float('inf')

                for geom in valid_intersections:
                    if geom.geom_type == 'Point':
                        pts = [geom]
                    elif geom.geom_type == 'MultiPoint':
                        pts = geom.geoms
                    else:
                        continue

                    for pt in pts:
                        dist = Point(p2).distance(pt)
                        if dist < min_dist:
                            min_dist = dist
                            closest_pt = pt

                if closest_pt:
                    if is_start:
                        new_coords[0] = (closest_pt.x, closest_pt.y)
                    else:
                        new_coords[-1] = (closest_pt.x, closest_pt.y)

    return LineString(new_coords)

gdf_grid_isolated['geometry'] = gdf_grid_isolated['geometry'].apply(
    lambda geom: extend_line_to_road(geom, gdf_roads_clean, roads_sindex, MAX_EXTEND_DIST)
)

# --- 5. MERGE AND SAVE ---
print("\nMerging isolated grid lines back into the road dataset...")

keep_cols = ['geometry', 'highway', 'is_grid']
roads_cols = [c for c in gdf_roads_clean.columns if c in keep_cols or c == 'geometry']
grid_cols = [c for c in gdf_grid_isolated.columns if c in keep_cols or c == 'geometry']

final_gdf = pd.concat([
    gdf_roads_clean[roads_cols],
    gdf_grid_isolated[grid_cols]
], ignore_index=True)

print(f"Total rows in final dataset: {len(final_gdf)}")

print("Projecting back to original CRS...")
final_gdf = final_gdf.to_crs(original_crs)

print(f"\nSaving final cleaned dataset to GeoPackage...")
final_gdf.to_file(OUTPUT_PATH, driver="GPKG")

print(f"Saved successfully to: {OUTPUT_PATH} ✅")

Loading stitched road network...
Separating osm_grid lines from the main road planarization...

Planarizing the network... (Fracturing lines exactly at intersections)
Recovering road attributes via midpoints...


KeyError: "['name'] not in index"

In [27]:
import geopandas as gpd
import pandas as pd

def apply_exclusions():
    # 1. Paths
    roads_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_clean.gpkg'
    exclusions_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/EXCLUSIONS/MA_exclusions.gpkg'

    # Saving to a new file so you don't accidentally overwrite your clean base data!
    output_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_fully_excluded.gpkg'

    # 2. Load Data
    print("Loading roads and exclusions...")
    # Using pyogrio to speed up loading the GPKG
    roads = gpd.read_file(roads_path, engine="pyogrio")
    exclusions = gpd.read_file(exclusions_path, engine="pyogrio")

    # 3. Project to meters (MA State Plane) for accurate length math
    print("Projecting to EPSG:26986 (meters) for accurate length calculations...")
    roads = roads.to_crs("EPSG:26986")
    exclusions = exclusions.to_crs("EPSG:26986")

    # 4. Spatial Join to find interacting roads (The Bounding Box Filter)
    print("Using Spatial Index to find roads that touch the exclusion zones...")
    roads_intersecting = gpd.sjoin(roads, exclusions, how='inner', predicate='intersects')
    roads_to_cut_idx = roads_intersecting.index.unique()

    print(f"Total roads: {len(roads):,}")
    print(f"Roads interacting with exclusions: {len(roads_to_cut_idx):,}")

    # 5. Split dataset into Safe vs. Needs Cutting
    safe_roads = roads[~roads.index.isin(roads_to_cut_idx)].copy()
    roads_to_cut = roads[roads.index.isin(roads_to_cut_idx)].copy()

    # ==========================================
    # 6. Clean and Union Exclusions
    # ==========================================
    print("Cleaning invalid geometries in exclusions (fixing self-intersections)...")
    # Step A: Force valid geometries
    exclusions['geometry'] = exclusions.geometry.make_valid()

    # Step B: Apply a 0-meter buffer (The ultimate silver bullet for TopologyExceptions)
    exclusions['geometry'] = exclusions.geometry.buffer(0)

    # Step C: Drop any polygons that collapsed into empty shapes during cleaning
    exclusions = exclusions[~exclusions.geometry.is_empty]

    print("Unioning exclusion zones (this prevents overlapping exclusions from causing weird geometry bugs)...")
    if hasattr(exclusions.geometry, 'union_all'):
        exclusion_union = exclusions.geometry.union_all()
    else:
        exclusion_union = exclusions.unary_union

    # 7. Cut and Measure!
    print("Applying cuts and checking the 50% overlap rule...")
    roads_to_cut['original_length'] = roads_to_cut.geometry.length

    # Cookie-cutter the road
    roads_to_cut['geometry'] = roads_to_cut['geometry'].difference(exclusion_union)

    # Measure what is left
    roads_to_cut['new_length'] = roads_to_cut.geometry.length

    # Calculate overlap percentage
    roads_to_cut['overlap_pct'] = (roads_to_cut['original_length'] - roads_to_cut['new_length']) / roads_to_cut['original_length']

    # 8. Enforce the 50% Rule
    failed_mask = roads_to_cut['overlap_pct'] > 0.50
    failed_count = failed_mask.sum()
    print(f"Discarding {failed_count:,} roads because >50% of their length was inside exclusions.")

    # Keep roads with <= 50% overlap, and drop completely empty geometries
    roads_to_cut = roads_to_cut[~failed_mask & ~roads_to_cut['geometry'].is_empty].copy()

    # Clean up calculation columns
    roads_to_cut = roads_to_cut.drop(columns=['original_length', 'new_length', 'overlap_pct'])

    # 9. Recombine and Save
    print("Recombining the road network...")
    final_roads = pd.concat([safe_roads, roads_to_cut], ignore_index=True)

    print("Projecting back to EPSG:4326 (Standard GPS coordinates)...")
    final_roads = final_roads.to_crs("EPSG:4326")

    print(f"Saving final excluded roads to {output_path}...")
    final_roads.to_file(output_path, driver="GPKG")
    print("Done! ✅")

if __name__ == "__main__":
    apply_exclusions()

Loading roads and exclusions...
Projecting to EPSG:26986 (meters) for accurate length calculations...
Using Spatial Index to find roads that touch the exclusion zones...
Total roads: 647,853
Roads interacting with exclusions: 179,780
Cleaning invalid geometries in exclusions (fixing self-intersections)...
Unioning exclusion zones (this prevents overlapping exclusions from causing weird geometry bugs)...
Applying cuts and checking the 50% overlap rule...
Discarding 176,145 roads because >50% of their length was inside exclusions.
Recombining the road network...
Projecting back to EPSG:4326 (Standard GPS coordinates)...
Saving final excluded roads to /Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_fully_excluded.gpkg...
Done! ✅


In [28]:
import geopandas as gpd
import pandas as pd
from collections import Counter

def process_road_network():
    # 1. Paths
    roads_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_fully_excluded.gpkg'
    parkings_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_parkings.geojson'
    cemeteries_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/PUBLIC/MA_cemeteries.geojson'
    output_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_final_excluded.gpkg'

    # 2. Load Roads and Project to Meters
    print("Loading roads...")
    roads = gpd.read_file(roads_path)

    print("Projecting to MA State Plane (meters) for accurate measurements...")
    roads = roads.to_crs("EPSG:26986")

    # ==========================================
    # STEP A: REMOVE SHORT DEAD-END STUMPS (TOPOLOGY FIX)
    # ==========================================
    print("\nAnalyzing network topology for dead-ends...")

    candidate_mask = roads['highway'].isin(['service', 'residential']) & (roads.geometry.length < 250)
    candidates = roads[candidate_mask].copy()
    print(f"Found {len(candidates):,} candidate short roads to check.")

    endpoints = candidates.geometry.boundary
    endpoints_gdf = gpd.GeoDataFrame(geometry=endpoints, crs=roads.crs).explode(index_parts=False)

    endpoints_gdf['endpoint_id'] = range(len(endpoints_gdf))
    endpoints_gdf['parent_road_id'] = endpoints_gdf.index
    endpoints_gdf['geometry'] = endpoints_gdf.geometry.buffer(0.1)

    print("Checking if endpoints physically touch other roads...")
    connections = gpd.sjoin(endpoints_gdf, roads, how='inner', predicate='intersects')

    valid_connections = connections[connections['parent_road_id'] != connections.index_right]
    connected_endpoint_ids = valid_connections['endpoint_id'].unique()
    endpoints_gdf['is_connected'] = endpoints_gdf['endpoint_id'].isin(connected_endpoint_ids)

    road_connectivity = endpoints_gdf.groupby('parent_road_id')['is_connected'].all()
    stump_ids = road_connectivity[road_connectivity == False].index

    roads = roads[~roads.index.isin(stump_ids)]
    print(f"Removed {len(stump_ids):,} short stump roads (kept T-junctions and through-streets)!")

    # ==========================================
    # STEP B: CUT ROADS INSIDE POLYGONS (TARGETED)
    # ==========================================
    print("\nLoading parking and cemetery polygons...")
    parkings = gpd.read_file(parkings_path, engine="pyogrio", use_arrow=True)
    cemeteries = gpd.read_file(cemeteries_path, engine="pyogrio", use_arrow=True)

    print("Aligning CRS and cleaning geometries...")
    parkings = parkings.to_crs(roads.crs)
    cemeteries = cemeteries.to_crs(roads.crs)

    # Combine exclusions
    exclusions = pd.concat([parkings[['geometry']], cemeteries[['geometry']]], ignore_index=True)

    # Wash geometries to prevent TopologyExceptions!
    exclusions['geometry'] = exclusions.geometry.make_valid()
    exclusions['geometry'] = exclusions.geometry.buffer(0)
    exclusions = exclusions[~exclusions.geometry.is_empty]

    print("Using Spatial Index to find the roads that actually touch exclusions...")
    roads_intersecting = gpd.sjoin(roads, exclusions, how='inner', predicate='intersects')
    interacting_idx = roads_intersecting.index.unique()

    # --- NEW LOGIC: Only target 'service' and 'residential' roads ---
    # Any other road type (like primary) will skip this entirely and remain "safe"
    target_mask = roads.index.isin(interacting_idx) & roads['highway'].isin(['service', 'residential'])

    print(f"Out of {len(roads):,} roads, {len(interacting_idx):,} interact with polygons.")
    print(f"However, only {target_mask.sum():,} are 'service'/'residential' and will be evaluated/cut.")

    # Split the dataset: safe roads (untouched) vs. roads we need to cut
    safe_roads = roads[~target_mask].copy()
    roads_to_cut = roads[target_mask].copy()

    print("Unioning exclusion zones for the final cut...")
    if hasattr(exclusions.geometry, 'union_all'):
        exclusion_union = exclusions.geometry.union_all()
    else:
        exclusion_union = exclusions.unary_union

    print("Erasing road segments and checking overlap percentage...")

    if len(roads_to_cut) > 0:
        roads_to_cut['original_length'] = roads_to_cut.geometry.length
        roads_to_cut['geometry'] = roads_to_cut['geometry'].difference(exclusion_union)
        roads_to_cut['new_length'] = roads_to_cut.geometry.length

        roads_to_cut['overlap_pct'] = (roads_to_cut['original_length'] - roads_to_cut['new_length']) / roads_to_cut['original_length']

        failed_roads_count = (roads_to_cut['overlap_pct'] > 0.50).sum()
        print(f"Discarding {failed_roads_count:,} roads because more than 50% of their length was inside an exclusion zone.")

        # Keep valid roads
        roads_to_cut = roads_to_cut[(roads_to_cut['overlap_pct'] <= 0.50) & (~roads_to_cut['geometry'].is_empty)]
        roads_to_cut = roads_to_cut.drop(columns=['original_length', 'new_length', 'overlap_pct'])

    print("Recombining the road network...")
    roads = pd.concat([safe_roads, roads_to_cut], ignore_index=True)

    # ==========================================
    # STEP C: SAVE
    # ==========================================
    print("\nProjecting back to standard GPS coordinates (EPSG:4326)...")
    roads = roads.to_crs("EPSG:4326")

    print(f"Saving final dataset to {output_path}...")
    roads.to_file(output_path, driver="GPKG")
    print("Done! ✅")

if __name__ == "__main__":
    process_road_network()


Loading roads...
Projecting to MA State Plane (meters) for accurate measurements...

Analyzing network topology for dead-ends...
Found 374,949 candidate short roads to check.
Checking if endpoints physically touch other roads...
Removed 216,509 short stump roads (kept T-junctions and through-streets)!

Loading parking and cemetery polygons...
Aligning CRS and cleaning geometries...
Using Spatial Index to find the roads that actually touch exclusions...
Out of 255,199 roads, 33,566 interact with polygons.
However, only 33,332 are 'service'/'residential' and will be evaluated/cut.
Unioning exclusion zones for the final cut...
Erasing road segments and checking overlap percentage...
Discarding 26,604 roads because more than 50% of their length was inside an exclusion zone.
Recombining the road network...

Projecting back to standard GPS coordinates (EPSG:4326)...
Saving final dataset to /Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_final_excluded.gpkg...
Done!

In [29]:
import geopandas as gpd
import pandas as pd
import networkx as nx

def prune_road_network():
    # 1. Paths
    input_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_final_excluded.gpkg'
    output_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_pruned.gpkg'

    # 2. Load and Project
    print("Loading excluded road network...")
    roads = gpd.read_file(input_path, engine="pyogrio")
    roads = roads.to_crs("EPSG:26986")

    # ==========================================
    # STEP 1: EXPLODE MULTI-PART GEOMETRIES
    # ==========================================
    # This splits rows where multiple disconnected lines are grouped together into one row.
    # It exposes the "floating" segments so we can evaluate and delete them individually.
    print(f"\nExploding multi-part lines... (Original rows: {len(roads):,})")
    roads = roads.explode(index_parts=False).reset_index(drop=True)
    print(f"After explosion: {len(roads):,} discrete road segments.")

    # ==========================================
    # STEP 2: SECOND PASS STUMP REMOVAL
    # ==========================================
    print("\nRunning second pass on dead-end stumps and floating lines...")
    candidate_mask = roads['highway'].isin(['service', 'residential']) & (roads.geometry.length < 200)
    candidates = roads[candidate_mask].copy()

    # Get endpoints and buffer slightly for T-junctions
    endpoints = candidates.geometry.boundary
    endpoints_gdf = gpd.GeoDataFrame(geometry=endpoints, crs=roads.crs).explode(index_parts=False)
    endpoints_gdf['endpoint_id'] = range(len(endpoints_gdf))
    endpoints_gdf['parent_road_id'] = endpoints_gdf.index
    endpoints_gdf['geometry'] = endpoints_gdf.geometry.buffer(0.1)

    # Check connections
    connections = gpd.sjoin(endpoints_gdf, roads, how='inner', predicate='intersects')
    valid_connections = connections[connections['parent_road_id'] != connections.index_right]

    connected_endpoint_ids = valid_connections['endpoint_id'].unique()
    endpoints_gdf['is_connected'] = endpoints_gdf['endpoint_id'].isin(connected_endpoint_ids)

    # A line is a stump (or floating) if NOT ALL of its endpoints connect
    road_connectivity = endpoints_gdf.groupby('parent_road_id')['is_connected'].all()
    stump_ids = road_connectivity[road_connectivity == False].index

    roads = roads[~roads.index.isin(stump_ids)].reset_index(drop=True)
    print(f"Removed {len(stump_ids):,} stumps and floating lines!")

    # ==========================================
    # STEP 3: REMOVE ISOLATED ROAD CLUSTERS
    # ==========================================
    print("\nBuilding network graph to find isolated road clusters...")

    # We buffer all roads by 10cm to ensure we catch T-junctions as "touching"
    roads_buffered = gpd.GeoDataFrame(geometry=roads.geometry.buffer(0.1), crs=roads.crs)

    # Spatial join the network against itself! This creates a list of every road that touches another road.
    intersections = gpd.sjoin(roads_buffered, roads, how='inner', predicate='intersects')

    # Build a Graph where Nodes = Roads, and Edges = Intersections
    G = nx.Graph()
    # Add an edge for every pair of roads that intersect
    G.add_edges_from(zip(intersections.index, intersections.index_right))

    # Find all distinct connected subgraphs (clusters)
    clusters = list(nx.connected_components(G))
    print(f"Found {len(clusters):,} distinct disconnected road networks.")

    # Keep only clusters that have at least 100 connected road segments.
    # (Why not just keep the single largest? Because MA has islands like Martha's Vineyard!
    # If we only keep the absolute biggest network, we'd delete the islands entirely.)
    MIN_CLUSTER_SIZE = 100

    valid_road_indices = []
    dropped_clusters = 0

    for cluster in clusters:
        if len(cluster) >= MIN_CLUSTER_SIZE:
            valid_road_indices.extend(cluster)
        else:
            dropped_clusters += 1

    roads = roads.loc[valid_road_indices].reset_index(drop=True)
    print(f"Dropped {dropped_clusters:,} isolated clusters (small disconnected road webs).")
    print(f"Final network contains {len(roads):,} perfectly connected road segments.")

    # ==========================================
    # STEP 4: SAVE
    # ==========================================
    print("\nProjecting back to GPS coordinates (EPSG:4326)...")
    roads = roads.to_crs("EPSG:4326")

    print(f"Saving pristine network to {output_path}...")
    roads.to_file(output_path, driver="GPKG")
    print("Done! ✅")

if __name__ == "__main__":
    prune_road_network()

Loading excluded road network...

Exploding multi-part lines... (Original rows: 228,595)
After explosion: 231,742 discrete road segments.

Running second pass on dead-end stumps and floating lines...
Removed 28,590 stumps and floating lines!

Building network graph to find isolated road clusters...
Found 2,897 distinct disconnected road networks.
Dropped 2,885 isolated clusters (small disconnected road webs).
Final network contains 195,477 perfectly connected road segments.

Projecting back to GPS coordinates (EPSG:4326)...
Saving pristine network to /Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_pruned.gpkg...
Done! ✅


In [35]:
import geopandas as gpd
import pandas as pd
import networkx as nx
from shapely.geometry import MultiLineString, Point
from shapely.ops import linemerge, substring
from collections import defaultdict

def merge_and_split_linestrings():
    # 1. Paths
    input_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_pruned.gpkg'
    output_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_stitched.gpkg'

    # 2. Load and Project
    print("Loading pruned roads...")
    roads = gpd.read_file(input_path, engine="pyogrio")
    print(f"Original total roads: {len(roads):,}")

    print("Projecting to MA State Plane (meters)...")
    roads = roads.to_crs("EPSG:26986")

    # ==========================================
    # STEP 1: MAP ALL ROAD CONNECTIONS
    # ==========================================
    print("\nMapping endpoints to find 2-degree connections...")
    pt_to_roads = defaultdict(list)

    for row in roads.itertuples():
        geom = row.geometry
        if geom is None or geom.is_empty or geom.geom_type != 'LineString':
            continue

        coords = list(geom.coords)
        pt_to_roads[coords[0]].append(row.Index)
        pt_to_roads[coords[-1]].append(row.Index)

    # ==========================================
    # STEP 2: BUILD THE MERGE GRAPH
    # ==========================================
    print("Identifying valid contiguous road segments...")
    G = nx.Graph()

    def names_match(n1, n2):
        if pd.isna(n1) and pd.isna(n2): return True
        return str(n1) == str(n2)

    for pt, connected_roads in pt_to_roads.items():
        if len(connected_roads) == 2:
            r1, r2 = connected_roads[0], connected_roads[1]
            if r1 == r2: continue

            hw1, hw2 = roads.at[r1, 'highway'], roads.at[r2, 'highway']
            name1 = roads.at[r1, 'name'] if 'name' in roads.columns else None
            name2 = roads.at[r2, 'name'] if 'name' in roads.columns else None

            # Always stitch osm_grid together ignoring names, otherwise enforce name match
            if hw1 == hw2:
                if hw1 == 'osm_grid':
                    G.add_edge(r1, r2)
                elif names_match(name1, name2):
                    G.add_edge(r1, r2)

    # ==========================================
    # STEP 3: STITCH AND ENFORCE 3000m LIMIT
    # ==========================================
    print("\nStitching roads (max 3500m limit per stitch, unlimited for osm_grid)...")
    roads_to_drop = set()
    new_roads_data = []

    components = list(nx.connected_components(G))

    for comp in components:
        subgraph = G.subgraph(comp)
        start_node = next((n for n, d in subgraph.degree() if d == 1), list(comp)[0])
        path_sequence = list(nx.dfs_preorder_nodes(subgraph, start_node))

        current_chunk = []
        current_len = 0.0

        # Check if this component is an osm_grid
        is_grid = (roads.at[path_sequence[0], 'highway'] == 'osm_grid')

        for r_idx in path_sequence:
            l = roads.at[r_idx, 'geometry'].length

            # If it's NOT a grid and exceeds length, break it. Otherwise, keep accumulating.
            if not is_grid and current_len + l > 3500 and len(current_chunk) > 0:
                if len(current_chunk) > 1:
                    roads_to_drop.update(current_chunk)
                    geoms = [roads.at[i, 'geometry'] for i in current_chunk]
                    merged_geom = linemerge(MultiLineString(geoms))

                    base_row = roads.loc[current_chunk[0]].copy()
                    base_row['geometry'] = merged_geom
                    new_roads_data.append(base_row)

                current_chunk = [r_idx]
                current_len = l
            else:
                current_chunk.append(r_idx)
                current_len += l

        if len(current_chunk) > 1:
            roads_to_drop.update(current_chunk)
            geoms = [roads.at[i, 'geometry'] for i in current_chunk]
            merged_geom = linemerge(MultiLineString(geoms))

            base_row = roads.loc[current_chunk[0]].copy()
            base_row['geometry'] = merged_geom
            new_roads_data.append(base_row)

    print(f"Sewed {len(roads_to_drop):,} fragments into {len(new_roads_data):,} pristine lines.")

    # Apply the stitched roads to a working dataset
    final_roads = roads.drop(index=list(roads_to_drop)).copy()
    if new_roads_data:
        new_roads_df = gpd.GeoDataFrame(new_roads_data, crs=roads.crs)
        final_roads = pd.concat([final_roads, new_roads_df], ignore_index=True)

    # ==========================================
    # STEP 4: SPLIT ROADS LONGER THAN 5KM
    # ==========================================
    print("\nScanning for massive roads (>5km) to split into <=3500m chunks...")
    final_roads['length'] = final_roads.geometry.length

    # Exclude osm_grid from being split here!
    long_roads_mask = (
        (final_roads['length'] > 5000) &
        (final_roads.geom_type == 'LineString') &
        (final_roads['highway'] != 'osm_grid')
    )

    long_roads = final_roads[long_roads_mask].copy()
    safe_roads = final_roads[~long_roads_mask].copy()

    print(f"Found {len(long_roads)} roads exceeding 5km. Processing...")

    if len(long_roads) > 0:
        # Get all true junctions (endpoints of every road currently in the network)
        # Because we stitched degree-2 nodes, endpoints are now degree 1, 3, 4, etc.
        all_endpoints = set()
        for geom in final_roads.geometry:
            if geom is not None and not geom.is_empty and geom.geom_type == 'LineString':
                coords = list(geom.coords)
                all_endpoints.add(coords[0])
                all_endpoints.add(coords[-1])

        split_roads_data = []

        for idx, row in long_roads.iterrows():
            line = row.geometry
            total_len = line.length

            # Find which coordinates on this long line are actual network junctions
            line_coords = set(line.coords)
            junc_coords_on_line = line_coords.intersection(all_endpoints)

            # Calculate how far down the line each junction is
            junc_distances = sorted([line.project(Point(c)) for c in junc_coords_on_line])

            current_start = 0.0

            while total_len - current_start > 3500:
                target_dist = current_start + 3500

                # Find best junction between [current_start + 500m] and [target_dist]
                # We add 500m so we don't accidentally snap to a junction 2 meters away and make a tiny stub
                valid_juncs = [d for d in junc_distances if current_start + 500 < d <= target_dist]

                # If there's an intersection in that range, split there! Otherwise, force a cut at 3500m.
                if valid_juncs:
                    split_dist = max(valid_juncs)
                else:
                    split_dist = target_dist

                # Extract the fragment using Shapely's clean substring tool
                fragment = substring(line, current_start, split_dist)

                new_row = row.copy()
                new_row['geometry'] = fragment
                split_roads_data.append(new_row)

                current_start = split_dist

            # Save the final remaining tail piece of the road
            if current_start < total_len:
                fragment = substring(line, current_start, total_len)
                new_row = row.copy()
                new_row['geometry'] = fragment
                split_roads_data.append(new_row)

        print(f"Successfully split {len(long_roads)} massive roads into {len(split_roads_data)} smaller logical fragments.")

        # Combine back into the main dataset
        split_df = gpd.GeoDataFrame(split_roads_data, crs=final_roads.crs)
        final_roads = pd.concat([safe_roads, split_df], ignore_index=True)

    # Clean up the temp length column
    final_roads = final_roads.drop(columns=['length'])

    # ==========================================
    # STEP 5: SAVE
    # ==========================================
    print("\nProjecting back to Standard GPS (EPSG:4326)...")
    final_roads = final_roads.to_crs("EPSG:4326")

    print(f"\n---> FINAL TOTAL ROADS INSIDE DATASET: {len(final_roads):,} <---")

    print(f"Saving finalized dataset to {output_path}...")
    final_roads.to_file(output_path, driver="GPKG")
    print("Done! ✅")

if __name__ == "__main__":
    merge_and_split_linestrings()

Loading pruned roads...
Original total roads: 195,477
Projecting to MA State Plane (meters)...

Mapping endpoints to find 2-degree connections...
Identifying valid contiguous road segments...

Stitching roads (max 3500m limit per stitch, unlimited for osm_grid)...
Sewed 67,157 fragments into 25,897 pristine lines.

Scanning for massive roads (>5km) to split into <=3500m chunks...
Found 175 roads exceeding 5km. Processing...
Successfully split 175 massive roads into 392 smaller logical fragments.

Projecting back to Standard GPS (EPSG:4326)...

---> FINAL TOTAL ROADS INSIDE DATASET: 154,434 <---
Saving finalized dataset to /Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_stitched.gpkg...
Done! ✅


In [40]:
import geopandas as gpd
import pandas as pd
import networkx as nx
from shapely.geometry import Point, MultiLineString, LineString
from shapely.ops import linemerge
from collections import defaultdict
import warnings

warnings.filterwarnings('ignore')

def heal_network(edges_gdf, protected_coords):
    """Helper function to zip collinear segments sharing a degree-2 node."""
    print("Stitching collinear degree-2 segments back together...")
    pt_to_roads = defaultdict(list)

    # Round protected coords for safe checking
    rounded_protected = {(round(x, 3), round(y, 3)) for x, y in protected_coords}

    for idx, row in edges_gdf.iterrows():
        coords = list(row.geometry.coords)
        pt_to_roads[(round(coords[0][0], 3), round(coords[0][1], 3))].append(idx)
        pt_to_roads[(round(coords[-1][0], 3), round(coords[-1][1], 3))].append(idx)

    G = nx.Graph()
    G.add_nodes_from(edges_gdf.index)

    for pt, connected_roads in pt_to_roads.items():
        if len(connected_roads) == 2:
            # CRITICAL: Do not zip this node if it's the exact spot an osm_grid connects!
            if pt in rounded_protected:
                continue

            r1, r2 = connected_roads[0], connected_roads[1]
            if r1 == r2: continue

            if 'highway' in edges_gdf.columns:
                hw1 = edges_gdf.at[r1, 'highway']
                hw2 = edges_gdf.at[r2, 'highway']
                if hw1 == hw2:
                    G.add_edge(r1, r2)
            else:
                G.add_edge(r1, r2)

    components = list(nx.connected_components(G))
    final_edges_data = []

    for comp in components:
        comp_list = list(comp)
        if len(comp_list) == 1:
            final_edges_data.append(edges_gdf.loc[comp_list[0]])
        else:
            geoms = [edges_gdf.at[i, 'geometry'] for i in comp_list]
            merged_geom = linemerge(MultiLineString(geoms))

            base_row = edges_gdf.loc[comp_list[0]].copy()

            if 'is_grid' in edges_gdf.columns:
                has_grid = any(edges_gdf.at[i, 'is_grid'] == 'yes' for i in comp_list)
                base_row['is_grid'] = 'yes' if has_grid else 'no'

            if merged_geom.geom_type == 'MultiLineString':
                for sub_geom in merged_geom.geoms:
                    new_row = base_row.copy()
                    new_row['geometry'] = sub_geom
                    final_edges_data.append(new_row)
            else:
                new_row = base_row.copy()
                new_row['geometry'] = merged_geom
                final_edges_data.append(new_row)

    healed_gdf = gpd.GeoDataFrame(final_edges_data, crs=edges_gdf.crs).reset_index(drop=True)
    print(f"Healed! Graph contains {len(healed_gdf):,} logical edges.")
    return healed_gdf


def build_perfect_topology():
    input_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_stitched.gpkg'
    edges_output = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_network_edges.gpkg'
    nodes_output = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_network_nodes.gpkg'
    cleaned_stitched_output = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_stitched_cleaned.gpkg'

    print("Loading stitched road network...")
    roads = gpd.read_file(input_path, engine="pyogrio")
    roads = roads.to_crs("EPSG:26986")
    roads['geometry'] = roads.geometry.make_valid()

    roads = roads.explode(index_parts=False).reset_index(drop=True)
    roads = roads[roads.geometry.type == 'LineString'].copy()

    print("Separating osm_grid lines...")
    if 'highway' in roads.columns:
        grid_mask = roads['highway'] == 'osm_grid'
        grid_gdf = roads[grid_mask].copy()
        roads = roads[~grid_mask].copy()
    else:
        grid_gdf = gpd.GeoDataFrame(columns=roads.columns, crs=roads.crs)

    # ==========================================
    # MAP LOGICAL CONNECTIONS FOR OSM_GRID
    # ==========================================
    print("\nMapping logical graph connections for osm_grid (preserving physical geometry)...")
    road_union = roads.unary_union
    grid_edges_data = []
    protected_coords = set()

    for idx, row in grid_gdf.iterrows():
        line = row.geometry
        intersection = line.intersection(road_union)

        points = []
        geom_list = getattr(intersection, 'geoms', [intersection]) if hasattr(intersection, 'geoms') else [intersection]

        for geom in geom_list:
            if geom.is_empty: continue
            if geom.geom_type == 'Point':
                points.append(geom)
            elif geom.geom_type == 'MultiPoint':
                points.extend(list(geom.geoms))
            elif geom.geom_type == 'LineString' and len(geom.coords) >= 2:
                points.extend([Point(geom.coords[0]), Point(geom.coords[-1])])
            elif geom.geom_type == 'MultiLineString':
                for sub in geom.geoms:
                    if len(sub.coords) >= 2:
                        points.extend([Point(sub.coords[0]), Point(sub.coords[-1])])

        start_pt = Point(line.coords[0])
        end_pt = Point(line.coords[-1])

        if not points:
            start_node_coord = (start_pt.x, start_pt.y)
            end_node_coord = (end_pt.x, end_pt.y)
        else:
            dist_to_pt = {line.project(pt): pt for pt in points}
            d_min, d_max = min(dist_to_pt.keys()), max(dist_to_pt.keys())

            if len(dist_to_pt) == 1 or abs(d_max - d_min) < 0.1:
                pt_exact = dist_to_pt[d_min]
                exact_coord = (pt_exact.x, pt_exact.y)

                if d_min < line.length / 2.0:
                    start_node_coord = exact_coord
                    end_node_coord = (end_pt.x, end_pt.y)
                else:
                    start_node_coord = (start_pt.x, start_pt.y)
                    end_node_coord = exact_coord

                protected_coords.add(exact_coord)
            else:
                pt_start = dist_to_pt[d_min]
                pt_end = dist_to_pt[d_max]

                start_node_coord = (pt_start.x, pt_start.y)
                end_node_coord = (pt_end.x, pt_end.y)

                protected_coords.add(start_node_coord)
                protected_coords.add(end_node_coord)

        new_row = row.copy()
        new_row['geometry'] = line
        new_row['logical_start'] = start_node_coord
        new_row['logical_end'] = end_node_coord
        grid_edges_data.append(new_row)

    grid_gdf_mapped = gpd.GeoDataFrame(grid_edges_data, crs=roads.crs)

    # ==========================================
    # PLANARIZE ROADS (Using Micro-Cutters!)
    # ==========================================
    print("\nPlanarizing roads and forcing fractures at grid connection points using micro-cutters...")

    # Create tiny 2mm lines at the exact connection coordinates to force a clean split
    cutter_lines = []
    for pt in protected_coords:
        px, py = pt[0], pt[1]
        cutter_lines.append(LineString([(px - 0.001, py), (px + 0.001, py)]))
        cutter_lines.append(LineString([(px, py - 0.001), (px, py + 0.001)]))

    if hasattr(roads.geometry, 'union_all'):
        noded_geom = roads.geometry.union_all()
    else:
        noded_geom = roads.unary_union

    geoms_to_node = [noded_geom] + cutter_lines
    noded_geom = gpd.GeoSeries(geoms_to_node).unary_union

    edges_list = [geom for geom in getattr(noded_geom, 'geoms', [noded_geom]) if geom.geom_type == 'LineString']
    exploded_edges = gpd.GeoDataFrame(geometry=edges_list, crs=roads.crs)

    # Vaporize the 2mm cutter debris! Keep only actual road segments.
    exploded_edges = exploded_edges[exploded_edges.geometry.length > 0.005].copy()

    print("Recovering road attributes via midpoints...")
    exploded_edges['midpoint'] = exploded_edges.geometry.interpolate(0.5, normalized=True)
    midpoints_gdf = gpd.GeoDataFrame(geometry=exploded_edges['midpoint'], crs=roads.crs)

    cols_to_keep = [c for c in ['highway', 'name', 'is_grid'] if c in roads.columns]
    joined = gpd.sjoin_nearest(midpoints_gdf, roads[cols_to_keep + ['geometry']], how='left')
    joined = joined[~joined.index.duplicated(keep='first')]

    for col in cols_to_keep:
        exploded_edges[col] = joined[col]

    exploded_edges = exploded_edges.drop(columns=['midpoint'])

    # ==========================================
    # HEALING & CLEANING
    # ==========================================
    print("\n--- PASS 1: HEALING ---")
    edges_gdf = heal_network(exploded_edges, protected_coords)

    print("\n--- PASS 2: HEALING ---")
    edges_gdf = heal_network(edges_gdf, protected_coords)

    # ==========================================
    # EXTRACT STRICT NODES
    # ==========================================
    print("\nExtracting precise coordinate nodes...")

    edges_gdf['logical_start'] = None
    edges_gdf['logical_end'] = None

    edges_combined = pd.concat([edges_gdf, grid_gdf_mapped], ignore_index=True)
    edges_combined['edge_id'] = [f"E_{i:06d}" for i in range(len(edges_combined))]

    coord_to_node_id = {}
    node_to_edges = defaultdict(list)
    node_to_neighbors = defaultdict(set)

    def get_exact_node_id(pt_coords):
        # Round to 3 decimals (1 millimeter) so roads and grids lock together perfectly
        rounded_coords = (round(pt_coords[0], 3), round(pt_coords[1], 3))
        if rounded_coords not in coord_to_node_id:
            coord_to_node_id[rounded_coords] = f"N_{len(coord_to_node_id):06d}"
        return coord_to_node_id[rounded_coords]

    edges_combined['start_node'] = None
    edges_combined['end_node'] = None

    for idx, row in edges_combined.iterrows():
        if pd.notna(row.get('logical_start')):
            u_coord = row['logical_start']
            v_coord = row['logical_end']
        else:
            coords = list(row.geometry.coords)
            u_coord = coords[0]
            v_coord = coords[-1]

        u_id = get_exact_node_id(u_coord)
        v_id = get_exact_node_id(v_coord)
        edge_id = row['edge_id']

        edges_combined.at[idx, 'start_node'] = u_id
        edges_combined.at[idx, 'end_node'] = v_id

        node_to_edges[u_id].append(edge_id)
        node_to_edges[v_id].append(edge_id)

        if u_id != v_id:
            node_to_neighbors[u_id].add(v_id)
            node_to_neighbors[v_id].add(u_id)

    print(f"Constructing final Nodes dataset ({len(coord_to_node_id):,} true nodes)...")
    nodes_data = []
    for coord_key, n_id in coord_to_node_id.items():
        nodes_data.append({
            'node_id': n_id,
            'connected_edges': ", ".join(node_to_edges[n_id]),
            'connected_nodes': ", ".join(sorted(list(node_to_neighbors[n_id]))),
            'geometry': Point(coord_key[0], coord_key[1]) # Saved accurately in State Plane
        })

    nodes_gdf = gpd.GeoDataFrame(nodes_data, crs=edges_combined.crs)

    final_cols = ['edge_id', 'start_node', 'end_node', 'geometry']
    for col in ['highway', 'name', 'is_grid']:
        if col in edges_combined.columns: final_cols.append(col)
    edges_combined = edges_combined[final_cols]

    print("\nProjecting back to GPS coordinates (EPSG:4326) and saving...")
    edges_combined = edges_combined.to_crs("EPSG:4326")
    nodes_gdf = nodes_gdf.to_crs("EPSG:4326")
    roads_cleaned = roads.to_crs("EPSG:4326")

    edges_combined.to_file(edges_output, driver="GPKG")
    nodes_gdf.to_file(nodes_output, driver="GPKG")
    roads_cleaned.to_file(cleaned_stitched_output, driver="GPKG")

    print("\nPerfect network topology successfully built and synced! ✅")

if __name__ == "__main__":
    build_perfect_topology()

Loading stitched road network...
Separating osm_grid lines...

Mapping logical graph connections for osm_grid (preserving physical geometry)...

Planarizing roads and forcing fractures at grid connection points using micro-cutters...
Recovering road attributes via midpoints...

--- PASS 1: HEALING ---
Stitching collinear degree-2 segments back together...
Healed! Graph contains 342,619 logical edges.

--- PASS 2: HEALING ---
Stitching collinear degree-2 segments back together...
Healed! Graph contains 342,584 logical edges.

Extracting precise coordinate nodes...
Constructing final Nodes dataset (250,428 true nodes)...

Projecting back to GPS coordinates (EPSG:4326) and saving...

Perfect network topology successfully built and synced! ✅


In [44]:
import geopandas as gpd
import pandas as pd
import numpy as np
import networkx as nx
from shapely.geometry import Point, LineString
from collections import defaultdict
from tqdm import tqdm
import itertools
import warnings

warnings.filterwarnings('ignore')

def inject_artificial_network():
    # 1. Paths
    exclusions_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/EXCLUSIONS/MA_exclusions.gpkg'
    edges_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_network_edges.gpkg'
    nodes_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_network_nodes.gpkg'
    roads_path = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_stitched_cleaned.gpkg'

    # Output paths
    edges_out = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_network_edges_synthetic.gpkg'
    nodes_out = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_network_nodes_synthetic.gpkg'
    roads_out = '/Users/kuba/PycharmProjects/grid-finder-USA/raw_data/MA/TRANSPORT/MA_roads_stitched_cleaned_synthetic.gpkg'

    # 2. Load Data
    print("Loading datasets...")
    exclusions = gpd.read_file(exclusions_path, engine="pyogrio").to_crs("EPSG:26986")
    print(f"-> Raw exclusions loaded from disk: {len(exclusions):,} rows.")

    edges = gpd.read_file(edges_path, engine="pyogrio").to_crs("EPSG:26986")
    nodes = gpd.read_file(nodes_path, engine="pyogrio").to_crs("EPSG:26986")
    roads = gpd.read_file(roads_path, engine="pyogrio").to_crs("EPSG:26986")

    print("Validating geometries...")
    exclusions['geometry'] = exclusions.geometry.make_valid()

    print("Exploding multiparts...")
    exclusions = exclusions.explode(index_parts=False).reset_index(drop=True)

    exclusions = exclusions[exclusions.geometry.type.isin(['Polygon', 'MultiPolygon'])].copy()
    print(f"-> Valid Polygons ready for seeding: {len(exclusions):,} rows.")

    art_nodes_data = []
    art_edges_data = []

    node_counter = 1
    edge_counter = 1

    poly_to_art_nodes = defaultdict(list)
    art_nodes_dict = {}

    # ==========================================
    # STEP 1: GENERATE ARTIFICIAL NODES
    # ==========================================
    print("\nGenerating artificial nodes inside exclusion zones...")
    np.random.seed(42)

    for idx, row in tqdm(exclusions.iterrows(), total=len(exclusions), desc="Seeding Nodes"):
        poly = row.geometry
        if poly is None or poly.is_empty:
            continue

        area_sq_meters = poly.area
        if area_sq_meters < 5000:
            continue

        n_nodes = max(1, int(area_sq_meters / 300_000))
        minx, miny, maxx, maxy = poly.bounds
        pts = []
        attempts = 0

        while len(pts) < n_nodes and attempts < (n_nodes * 100):
            p = Point(np.random.uniform(minx, maxx), np.random.uniform(miny, maxy))
            if poly.contains(p):
                pts.append(p)
            attempts += 1

        for p in pts:
            nid = f"N_ART_{node_counter:06d}"
            art_nodes_data.append({'node_id': nid, 'geometry': p})
            poly_to_art_nodes[idx].append(nid)
            art_nodes_dict[nid] = p
            node_counter += 1

    if not art_nodes_data:
        print("No exclusion zones were large enough to require artificial nodes. Exiting.")
        return

    print(f"Generated {len(art_nodes_data):,} artificial nodes.")
    art_nodes_gdf = gpd.GeoDataFrame(art_nodes_data, crs=nodes.crs)

    # ==========================================
    # STEP 2: CONNECT INTERNAL ARTIFICIAL NODES (DENSER GRID)
    # ==========================================
    print("\nBuilding internal road networks inside the exclusion zones...")
    import itertools

    for poly_idx, nids in poly_to_art_nodes.items():
        if len(nids) < 2:
            continue

        # Create a temporary graph of ALL possible connections
        G = nx.Graph()
        for n1, n2 in itertools.combinations(nids, 2):
            p1, p2 = art_nodes_dict[n1], art_nodes_dict[n2]
            dist = p1.distance(p2)
            G.add_edge(n1, n2, weight=dist)

        # 1. Get the MST
        mst = nx.minimum_spanning_tree(G)
        dense_edges = set(mst.edges())

        # 2. ADD DENSITY
        for n in nids:
            neighbors = sorted([(v, data['weight']) for v, data in G[n].items()], key=lambda x: x[1])
            for v, dist in neighbors[:4]:
                edge = tuple(sorted((n, v)))
                dense_edges.add(edge)

        # 3. Draw the lines with a STRICT length limit!
        for u, v in dense_edges:
            dist = art_nodes_dict[u].distance(art_nodes_dict[v])

            # HARD CAP: Never draw an internal artificial edge longer than 2,000 meters (2km).
            # This prevents mega-polygons from shooting lines across the state.
            if dist > 2000:
                continue

            eid = f"E_ART_{edge_counter:06d}"
            line = LineString([art_nodes_dict[u], art_nodes_dict[v]])
            art_edges_data.append({
                'edge_id': eid, 'start_node': u, 'end_node': v,
                'highway': 'artificial', 'name': 'Exclusion Internal Link',
                'geometry': line
            })
            edge_counter += 1

    # ==========================================
    # STEP 3: HOOK THE REAL WORLD TO THE SYNTHETIC WORLD (CONTEXT-AWARE)
    # ==========================================
    print("\nHooking existing road network to the new synthetic grids...")

    # 1. Buffer the active exclusions by 50 meters to create a "catchment zone" around the boundaries
    active_exclusions = exclusions.loc[list(poly_to_art_nodes.keys())].copy()
    active_exclusions['geometry'] = active_exclusions.geometry.buffer(50)

    # 2. Find all real-world nodes standing inside these catchment zones.
    # By using sjoin, 'index_right' tells us EXACTLY which polygon the node is touching!
    nearby_nodes = gpd.sjoin(nodes, active_exclusions, how='inner', predicate='intersects')

    print(f"Found {len(nearby_nodes):,} real-world nodes right next to exclusion boundaries.")

    created_bridges = set()

    for idx, row in nearby_nodes.iterrows():
        orig_nid = row['node_id']
        poly_idx = row['index_right'] # The specific polygon this node is next to
        orig_geom = row['geometry']

        # 3. Get the artificial nodes belonging ONLY to this specific polygon
        local_art_nids = poly_to_art_nodes.get(poly_idx, [])
        if not local_art_nids:
            continue

        # 4. Find the absolute closest artificial node INSIDE this specific polygon
        best_art_nid = None
        min_dist = float('inf')

        for art_nid in local_art_nids:
            art_geom = art_nodes_dict[art_nid]
            dist = orig_geom.distance(art_geom)
            if dist < min_dist:
                min_dist = dist
                best_art_nid = art_nid

        if best_art_nid:
            # We allow the bridge to reach up to 1000 meters into the park.
            # This connects deep internal nodes without risking cross-state bridges.
            if min_dist > 1000:
                continue

            # Prevent drawing the exact same bridge twice if buffers overlap
            bridge_sig = tuple(sorted((orig_nid, best_art_nid)))
            if bridge_sig in created_bridges:
                continue

            created_bridges.add(bridge_sig)

            eid = f"E_ART_{edge_counter:06d}"
            line = LineString([orig_geom, art_nodes_dict[best_art_nid]])

            art_edges_data.append({
                'edge_id': eid, 'start_node': orig_nid, 'end_node': best_art_nid,
                'highway': 'artificial_bridge', 'name': 'Exclusion Entry Link',
                'geometry': line
            })
            edge_counter += 1
    # ==========================================
    # STEP 4: UPDATE GRAPH RELATIONSHIPS
    # ==========================================
    print(f"\nCreated total {len(art_edges_data):,} new synthetic road segments. Updating node topologies...")

    node_to_edges = {}
    node_to_neighbors = {}

    for _, row in nodes.iterrows():
        nid = row['node_id']
        node_to_edges[nid] = set(row['connected_edges'].split(', ')) if pd.notna(row['connected_edges']) and row['connected_edges'] else set()
        node_to_neighbors[nid] = set(row['connected_nodes'].split(', ')) if pd.notna(row['connected_nodes']) and row['connected_nodes'] else set()

    for nid in art_nodes_dict.keys():
        node_to_edges[nid] = set()
        node_to_neighbors[nid] = set()

    for edge in art_edges_data:
        u, v, eid = edge['start_node'], edge['end_node'], edge['edge_id']
        node_to_edges[u].add(eid)
        node_to_edges[v].add(eid)
        node_to_neighbors[u].add(v)
        node_to_neighbors[v].add(u)

    def get_edges_str(nid): return ", ".join(sorted(list(node_to_edges[nid])))
    def get_neighbors_str(nid): return ", ".join(sorted(list(node_to_neighbors[nid])))

    nodes['connected_edges'] = nodes['node_id'].apply(get_edges_str)
    nodes['connected_nodes'] = nodes['node_id'].apply(get_neighbors_str)

    art_nodes_gdf['connected_edges'] = art_nodes_gdf['node_id'].apply(get_edges_str)
    art_nodes_gdf['connected_nodes'] = art_nodes_gdf['node_id'].apply(get_neighbors_str)

    # ==========================================
    # STEP 5: MERGE AND SAVE EVERYTHING
    # ==========================================
    print("\nMerging artificial datasets into the master layers...")

    final_nodes = pd.concat([nodes, art_nodes_gdf], ignore_index=True)
    art_edges_gdf = gpd.GeoDataFrame(art_edges_data, crs=edges.crs)
    final_edges = pd.concat([edges, art_edges_gdf], ignore_index=True)

    art_roads_data = art_edges_gdf[['highway', 'name', 'geometry']].copy()
    art_roads_data['stitched_id'] = [eid.replace('E_ART_', 'R_ART_') for eid in art_edges_gdf['edge_id']]
    final_roads = pd.concat([roads, art_roads_data], ignore_index=True)

    print("\nProjecting to GPS Coordinates (EPSG:4326) and saving...")
    final_nodes = final_nodes.to_crs("EPSG:4326")
    final_edges = final_edges.to_crs("EPSG:4326")
    final_roads = final_roads.to_crs("EPSG:4326")

    final_edges.to_file(edges_out, driver="GPKG")
    final_nodes.to_file(nodes_out, driver="GPKG")
    final_roads.to_file(roads_out, driver="GPKG")

    print("\nSynthetic network completely integrated! ✅")

if __name__ == "__main__":
    inject_artificial_network()

Loading datasets...
-> Raw exclusions loaded from disk: 1 rows.
Validating geometries...
Exploding multiparts...
-> Valid Polygons ready for seeding: 1 rows.

Generating artificial nodes inside exclusion zones...


Seeding Nodes: 100%|██████████| 1/1 [00:01<00:00,  1.34s/it]


Generated 4,621 artificial nodes.

Building internal road networks inside the exclusion zones...

Hooking existing road network to the new synthetic grids...
Found 1,862 real-world nodes right next to exclusion boundaries.

Created total 13,181 new synthetic road segments. Updating node topologies...

Merging artificial datasets into the master layers...

Projecting to GPS Coordinates (EPSG:4326) and saving...

Synthetic network completely integrated! ✅


In [ ]:

# remove polygons early on